<a href="https://colab.research.google.com/github/loulakanani/machine-learning-in-finance/blob/main/Credit_Risk_Analysis_and_Pipeline_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Credit Risk Analysis and Pipeline Engineering: Investigating Default Behavior and Risk Drivers in High-Limit Outliers**

In [120]:
#1st question: A short financial question written in one paragraph.

How does default behavior differ between borrowers with low credit limits and those with extremely high credit limits (outliers)? Furthermore, is the risk associated with high credit limits a result of excessive borrowing (high BILL_AMT) or poor liquidity management (low PAY_AMT relative to the debt)?

In [121]:
#2 Load the complete course dataset and document the UCI source, licence or citation, observation unit, target, and variable meanings.

**DATA DOCUMENTATION**
*   **Dataset Source**: UCI Machine Learning Repository - Default of Credit Card Clients
*   **Licence**: This dataset is licensed under a Creative Commons Attribution 4.0 International (CC BY 4.0) license. This allows for the sharing and adaptation of the datasets for any purpose, provided that the appropriate credit is given.
*   **Observation Unit**: one credit-card account record for one borrower
*   **Target Variable**: default_next_month (1 = default next month, 0 = no default)
*   **

**VARIABLE MEANINGS**
*  **ID**:	Borrower identifier
*  **LIMIT_BAL**:	Credit limit in New Taiwan dollars
*  **SEX, EDUCATION, MARRIAGE, AGE**: Demographic features
*  **PAY_0 to PAY_6**: Repayment-status codes for earlier months
*  **BILL_AMT1–BILL_AMT6**: Bill statement amounts for six months
*  **PAY_AMT1–PAY_AMT6**:	Previous payment amounts for six months
*  **default_next_month**:	Default payment next mont


In [122]:
# Load the dataset and the libraries and rename the target column
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

credit = pd.read_excel("default of credit card clients.xls", header=1)
credit = credit.rename(columns={'default payment next month': 'default_next_month'})


In [123]:
#3 Show shape, dtypes, missingness, duplicate count, unique-ID count, target balance, and coded-category frequencies

In [124]:
print("shape:", credit.shape)

audit = pd.DataFrame({
    'dtype': credit.dtypes.astype(str),     #find the data type and then convert it to string
    'missing': credit.isna().sum(),         #find the missing value of every column and sum them up
    'unique': credit.nunique(),             #calculate the count of unique variables
    'min': credit.min(numeric_only=True),   #find the minimum value of the numeric variables
    'max': credit.max(numeric_only=True),   #find the maximum vaue of the numeric variables
    })
audit.T


shape: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_next_month
dtype,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,...,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64
missing,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
unique,30000,81,2,7,4,56,11,11,11,11,...,21548,21010,20604,7943,7899,7518,6937,6897,6939,2
min,1,10000,1,0,0,21,-2,-2,-2,-2,...,-170000,-81334,-339603,0,0,0,0,0,0,0
max,30000,1000000,2,6,3,79,8,8,8,8,...,891586,927171,961664,873552,1684259,896040,621000,426529,528666,1


In [125]:
print(credit.duplicated().sum())                    #check how duplicate the dataset has and then sum them up
print(credit['ID'].nunique())
print(credit["default_next_month"].value_counts())  #check the target balance

0
30000
default_next_month
0    23364
1     6636
Name: count, dtype: int64


Out of 30000, 6636 defaulted on their payments.


In [126]:
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]
for col in cat_cols:                                  #takes one by one column column
    print(col)
    print(credit[col].value_counts(dropna=False))     #counts how many times each value appears without ignoring the missing values

SEX
SEX
2    18112
1    11888
Name: count, dtype: int64
EDUCATION
EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64
MARRIAGE
MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64


In [127]:
#4 Create one controlled missingness example in memory. Explain whether the response is imputation, an indicator, removal, or a combination, and why.

In [128]:
demo = credit[['LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1']].head(8).copy()   #creates a copy of the first 8 rows of each selected columns.
demo.loc[[1, 4], 'PAY_AMT1'] = np.nan                                  #takes the 1st and the 4th row of column "PAY_ANT1" and replace the value with nan.
missing_rate = demo.isna().mean()                                      #calculate the mean of the missing values and save the result in a new variable.
print(demo)


   LIMIT_BAL  BILL_AMT1  PAY_AMT1
0      20000       3913       0.0
1     120000       2682       NaN
2      90000      29239    1518.0
3      50000      46990    2000.0
4      50000       8617       NaN
5      50000      64400    2500.0
6     500000     367965   55000.0
7     100000      11876     380.0


In [129]:
print(demo["PAY_AMT1"].median())                                       #calculate the median of the 'PAY_AMT1' and show.
demo['PAY_AMT1_was_missing'] = demo['PAY_AMT1'].isna().astype(int)     #convert missing values to int data type and save this in a new variable called 'PAY_AMT1_was_missing'. if its nan then return 1=True.
median_pay = demo['PAY_AMT1'].median()                                 #find the median of the initial column 'PAY_AMT1'.
demo['PAY_AMT1'] = demo['PAY_AMT1'].fillna(median_pay)                 #fill in the missing values with the median value.

display(demo[['PAY_AMT1', 'PAY_AMT1_was_missing']])                    #display the two columns.

1759.0


,PAY_AMT1,PAY_AMT1_was_missing
0,0.0,0
1,1759.0,1
2,1518.0,0
3,2000.0,0
4,1759.0,1
5,2500.0,0
6,55000.0,0
7,380.0,0


The approach combines imputation with an indicator. Imputing with the median ensures the column contains a valid value, allowing it to be processed by numerical transformers. However, for this specific research question regarding liquidity management, including an indicator is essential. The absence of payment data (PAY_AMT) may, in itself, carry critical information about the borrower's credit behavior. The indicator preserves this functional signal within the model.

In [130]:
#5 Create a duplicate-detection example and explain why an identifier is not automatically evidence that two rows are economically different.

In [131]:
small_subset = credit[['ID', 'LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1']].head(3).copy()  #Create a small copy focusing on variables relevant to the research question
demo_dups = pd.concat([small_subset, small_subset.iloc[[0]]], ignore_index=True)    #Create an exact duplicate of the 1st row and concatenate them to form the demo dataframe
print(demo_dups.shape)
print(demo_dups.duplicated().sum())

(4, 4)
1


A unique identifier (ID) does not automatically prove that two rows represent economically different events. For instance, if an automated system generates a new ID during an erroneous import, the exact same financial event might appear multiple times under different IDs

In [132]:
#6 Apply one documented outlier flag to a numeric variable. Report the threshold and count before deciding whether any treatment is appropriate.

In [133]:
lo = credit['LIMIT_BAL'].quantile(0.01)                                                  #Calculate the 1% and 99% quantiles for LIMIT_BAL
hi = credit['LIMIT_BAL'].quantile(0.99)

credit['LIMIT_BAL_outlier'] = ((credit['LIMIT_BAL'] < lo) | (credit['LIMIT_BAL'] > hi))   #Create a boolean flag for outliers

print(lo, hi)
print(credit['LIMIT_BAL_outlier'].value_counts())

10000.0 500000.0
LIMIT_BAL_outlier
False    29794
True       206
Name: count, dtype: int64


Out of the 30,000 records, 206 are outliers.

In [134]:
#7 Separate numeric, categorical, identifier, and target columns.

In [135]:
id_col = ['ID']
target_col = 'default next month'

cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']                    #define the categorical columns
num_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',        #define the numerical columns
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

x = credit[num_cols + cat_cols]
y = credit['default_next_month']

print(x.shape)
print(y.shape)

(30000, 23)
(30000,)


In [136]:
#8 Split the data with a fixed random seed and target stratification.

In [137]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=50, stratify=y)   #

print(x_train.shape)
print(x_test.shape)

(24000, 23)
(6000, 23)


 Splits the data into a training set (80%) to teach the model and a testing set (20%) to evaluate it, using a fixed seed for reproducible results and keeping the same percentage of defaults in both sets.

In [138]:
#9 Build a ColumnTransformer with a numeric Pipeline and a categorical Pipeline. Use training-only fitting and handle_unknown="ignore".

In [139]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),                              #replaces missing values ​​with the median.
    ("scale", StandardScaler())                                                 #It takes these values ​​and adjusts them to be close to zero.
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),                       #fills the missing values ​​with the most frequent word.
    ("onehot", OneHotEncoder(handle_unknown="ignore"))                          #converts the words into columns (0, 1), and if a new value appears, it is assigned the value 0.
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),                                            #places the numerical values ​​into the numeric pipe
    ("cat", categorical_pipe, cat_cols)                                         #places the categorical values ​​into the categorical pipe
])

In [140]:
x_train_transformed = preprocessor.fit_transform(x_train)   #training-only fitting

X_test_transformed = preprocessor.transform(x_test)         #Transform the test data WITHOUT fitting

In [141]:
#10 Display transformed shapes and feature names. Explain in plain language how the workflow prevents leakage.

In [142]:
print("Τελικό σχήμα X_train_ready:", x_train_transformed.shape)
print("Τελικό σχήμα X_test_ready:", X_test_transformed.shape)

feature_names = preprocessor.get_feature_names_out()
print(feature_names[:10])

Τελικό σχήμα X_train_ready: (24000, 91)
Τελικό σχήμα X_test_ready: (6000, 91)
['num__LIMIT_BAL' 'num__AGE' 'num__BILL_AMT1' 'num__BILL_AMT2'
 'num__BILL_AMT3' 'num__BILL_AMT4' 'num__BILL_AMT5' 'num__BILL_AMT6'
 'num__PAY_AMT1' 'num__PAY_AMT2']


In [143]:
# Data Leakage Prevention Explanation

The data split into Training and Testing sets was performed before any data transformation. In Question 9, the .fit_transform() method was applied exclusively to x_train. This ensures that all statistical parameters (such as the median for missing values and the mean/standard deviation for scaling) were calculated solely from the training data. The X_test data was only transformed using .transform(), meaning no information from the test set leaked into the model preparation phase, preventing overfitting and optimistic bias.


In [144]:
# ANALYSIS REGARDING THE RESEARCH QUESTION

In [145]:
#Definition of High Outliers
low_limit_threshold = credit['LIMIT_BAL'].quantile(0.01)
high_limit_threshold = credit['LIMIT_BAL'].quantile(0.99)

credit['limit_group'] = 'Medium'
credit.loc[credit['LIMIT_BAL'] <= low_limit_threshold, 'limit_group'] = 'Low'
credit.loc[credit['LIMIT_BAL'] >= high_limit_threshold, 'limit_group'] = 'Extremely High'

#Default Behavior
print(credit.groupby('limit_group')['default_next_month'].mean())

limit_group
Extremely High    0.112069
Low               0.399594
Medium            0.221666
Name: default_next_month, dtype: float64


In [146]:
#Excessive Borrowing vs Poor Liquidity για τους Outliers
credit['utilization_rate'] = credit['BILL_AMT1'] / credit['LIMIT_BAL']
credit['payment_ratio'] = np.where(credit['BILL_AMT1'] > 0, credit['PAY_AMT1'] / credit['BILL_AMT1'], np.nan)

#We filter only the outliers that defaulted, in order to see what went wrong
outliers_default = credit[(credit['limit_group'] == 'Extremely High') & (credit['default_next_month'] == 1)]
outliers_all = credit[credit['limit_group'] == 'Extremely High']

#Calculation of averages
util_all = round(outliers_all['utilization_rate'].mean(), 2)
util_def = round(outliers_default['utilization_rate'].mean(), 2)

pay_all = round(outliers_all['payment_ratio'].mean(), 2)
pay_def = round(outliers_default['payment_ratio'].mean(), 2)


print("Mean Utilization Rate for all outliers:", util_all)
print("Mean Utilization Rate for those who Defaulted:", util_def)

print("Mean Payment Ratio for all outliers:", pay_all)
print("Mean Payment Ratio for those who Defaulted:", pay_def)

Mean Utilization Rate for all outliers: 0.2
Mean Utilization Rate for those who Defaulted: 0.27
Mean Payment Ratio for all outliers: 1.5
Mean Payment Ratio for those who Defaulted: 1.63


The analysis reveals that default among high credit limit outliers is neither driven by excessive borrowing nor poor liquidity. Defaulted outliers maintain a very low utilization rate (0.27), meaning they do not over-utilize their credit limits. Furthermore, their high payment ratio (1.63) indicates they possess strong liquidity and actively overpay their current bills. Consequently, their default risk cannot be attributed to typical financial distress or cash-flow shortages, suggesting potential strategic defaults or sudden structural financial shifts.